# Café Analytics - Descriptive Analysis

In [ ]:
import pandas as pd
import warnings
import numpy as np
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.proportion import proportion_confint
import import_ipynb
import Customer_Churn_Analysis_MX
from IPython.display import display

In [ ]:
# Ignore all warnings in output
warnings.filterwarnings('ignore')

In [ ]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

## Revenue Trend Analysis

In [ ]:
# Load the cleaned full data parquet file into a pandas DataFrame
df = pd.read_parquet(
    './cafe_processed_files/cafe_full_mx.parquet',
)

In [ ]:
df_revenue = df[
    ['order_id', 'customer_id', 'order_price', 'order_time']
].drop_duplicates().reset_index(drop=True)

In [ ]:
df_revenue.head()

In [ ]:
df_revenue['order_year'] = (
    df_revenue['order_time'].dt.year
)
df_revenue['order_month'] = (
    df_revenue['order_time'].dt.month
)
df_revenue['order_day'] = (
    df_revenue['order_time'].dt.day
)
df_revenue['order_dow'] = (
    df_revenue['order_time'].dt.dayofweek
)
df_revenue['order_hour'] = (
    df_revenue['order_time'].dt.hour
)

In [ ]:
df_revenue['FY'] = (
    df_revenue.apply(
        lambda x: 'FY' + str(x['order_year'])[-2:] if x['order_month'] < 7
        else 'FY' + str(x['order_year'] + 1)[-2:],
        axis=1,
    )
)

In [ ]:
df_revenue.head()

### Monthly Revenue Trend

In [ ]:
monthly = (
    df_revenue[
        ['order_year', 'order_month', 'order_price']
    ].groupby(['order_year', 'order_month']).sum()
    .reset_index(drop=False)
)

In [ ]:
monthly.head()

In [ ]:
# Plot monthly revenue trend
fig, axes = plt.subplots(nrows=2, figsize=(10, 12))

for year in sorted(monthly['order_year'].unique()):
    month_data = monthly[
        monthly['order_year'] == year
    ]
    Y = month_data['order_price']
    X = month_data['order_month']
    
    axes[1].plot(X, Y, label=year, marker='.')
    axes[0].bar(
        x=year,
        height=month_data['order_price'].sum(), 
        label=year,
    )

axes[0].set_title('Online Sales Trend Over Years')
axes[0].legend(loc='upper left', bbox_to_anchor=(1.05, 1))
axes[0].set_xlabel('')
axes[0].set_ylabel('Online Sales (AUD)')
axes[1].set_title('Online Sales Trends by Years and Months')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Online Sales (AUD)')
axes[1].set_xticks(range(1, 13, 1))
plt.show()

* The bar chart above illustrates a **consistent rise** in business online sales from 2020 to 2023.
* Online sales in 2020 and 2021 consistently outperformed their corresponding previous year across all months, showing a notable increase during the winter season (June to August) and peaking in September before declining.
* In 2022, online sales generally stabilized over the course of the year but stayed higher than in 2021, except during the period from August to October. 
* Similary, online sales in 2023 remained stable overall but generally exceeded those of 2022 throughout the year.
* However, neither 2022 nor 2023 reached the September peak achieved in 2020 and 2021.

Question to investigate:<br><br>
**1) What factors contributed to the surge in sales from June to September in 2020 and 2021?**<br>
**2) Why did sales in 2022 and 2023 fail to reach the same peak in September?**<br>

Possible reasons: 

The **COVID pandemic lockdowns** likely boosted online sales, making the 2020 and 2021 figures more reflective of total sales (including both online and in-store). In contrast, the stablized sales in 2022 and 2023 may represent online sales only, as in-store shopping resumed.

As seen in the 2020 and 2021 figures, the café's online sales typically rise during the colder months from June to September in Australia, followed by a decline as the weather warms.

**TBC...**

### Most Profitable Day of the Week

In [ ]:
total_revenue_per_day = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_dow', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_dow'
    ]).sum().reset_index(drop=False)
)

total_revenue_per_day.head()

In [ ]:
dow_revenue = (
    total_revenue_per_day[
        ['order_dow', 'order_price']
    ].groupby('order_dow').mean()
    .reset_index(drop=False)
)

dow_revenue

In [ ]:
day_of_week = {
    0: 'Mon',
    1: 'Tue',
    2: 'Wed',
    3: 'Thu',
    4: 'Fri',
    5: 'Sat',
    6: 'Sun',
}

In [ ]:
# Plot average daily revenue over a week
fig, ax = plt.subplots(figsize=(6, 4))

dow_revenue.plot.bar(
    x='order_dow', y='order_price', 
    color='#D3D0C9', ax=ax
)

# Highlight the most profitable day of the week
top_dow = dow_revenue[
    (
        dow_revenue['order_price'] 
        == dow_revenue['order_price'].max() 
    )
]
ax.bar(
    x=top_dow['order_dow'],
    height=top_dow['order_price'],
    color='orange',
    width=0.5,
)

plt.xticks(
    ticks=list(day_of_week.keys()),
    labels=list(day_of_week.values()), 
    rotation=0,
)
plt.xlabel('')
ax.get_legend().remove()
ax.set_ylabel('Online Sales (AUD)')
plt.show()

As shown in the bar charts above, the average daily online sales generally remained steady from Mondays to Thursdays, but rose higher during Fridays and weekends, with a peak on **Saturdays** and **Sundays**.

### Peak Revenue Hours

In [ ]:
df_revenue.head()

In [ ]:
# Group sales revenue by year, month, day and hour
total_revenue_per_hour = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_hour', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_hour'
    ]).sum().reset_index(drop=False)
)

In [ ]:
print(total_revenue_per_hour.shape)
total_revenue_per_hour.head()

In [ ]:
# Display operating hours
sorted(total_revenue_per_hour['order_hour'].unique())

Some of order-taking hours seem unusual for a café. Let's check whether there are any outliers (irregular ordering hours) among them.

#### Identify and Remove Outliers

In [ ]:
# Plot the KDE distribution of records across operating hours
fig, ax = plt.subplots()
sns.kdeplot(
    x=total_revenue_per_hour['order_hour'],
    bw_adjust=1.6,  
    color='gray',
    ax=ax,
)
ax.set_xlabel('Order Hour')
ax.set_xticks(
    range(
        0,
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

According to the Kernel Density Estimation (KDE) plot above, the ordering hours in most days are between 5AM to 3PM. Let's confirm this using a boxplot and identify any potential outliers.

In [ ]:
# Draw a boxplot to identify outliers of order hours
fig, ax = plt.subplots()
sns.boxplot(
    y=total_revenue_per_hour['order_hour'],
    whis=1.5,    # Set the whiskers to extend up to 1.5 times the interquartile range (IQR)
    ax=ax,
)
plt.title("Data Point Distribution of Order Hours")
plt.ylabel('Operating Hour')
plt.yticks(
    range(
        total_revenue_per_hour['order_hour'].min(),
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

As shown in the boxplot above, it appears that the ordering hours extending past 5 PM have been identified as outliers.

In [ ]:
# Calculate the 1st quartile
Q1 = total_revenue_per_hour['order_hour'].quantile(0.25)
# Calculate the 3rd quartile
Q3 = total_revenue_per_hour['order_hour'].quantile(0.75)
# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower bound where the lower whisker ends
lower_bound = Q1 - 1.5 * IQR
# Calculate the upper bound where the upper whisker ends
upper_bound = Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

Data points falling below the lower bound or above the upper bound are considered **outliers**. Let's apply the lower and upper bounds to filter out irregular ordering hours, which only occurred on a few days in the business history, and determine the actual online ordering hours.

In [ ]:
# Filter out irregular ordering hours
total_revenue_per_hour = total_revenue_per_hour[
    (total_revenue_per_hour['order_hour'] >= lower_bound)
    & (total_revenue_per_hour['order_hour'] <= upper_bound)
]

In [ ]:
# Print the actual online ordering hours
sorted(total_revenue_per_hour['order_hour'].unique())

After filtering out the irregular ordering hours, we can see that the café typically accepts orders from **5 AM to 5 PM** on regular business days.

In [ ]:
# Calculate the average online sales revenue per hour
hourly_revenue = (
    total_revenue_per_hour[
        ['order_hour', 'order_price']
    ].groupby('order_hour').mean()
    .reset_index(drop=False)
)

hourly_revenue

In [ ]:
# Calculate the average revenue per hour across all days
avg_revenue_per_hour = total_revenue_per_hour['order_price'].mean()
print(avg_revenue_per_hour)

In [ ]:
# Plot average hourly revenue across hours
plt.bar(
    x=hourly_revenue['order_hour'],
    height=hourly_revenue['order_price'],
    color='#D3D0C9',
)
# Plot a constant line representing average revenue per 
# hour across all days
plt.plot(
    range(0, 24, 1),
    [avg_revenue_per_hour] * 24,
    linestyle='--',
    linewidth=0.5,
    c='black',
)

# Highlight the peak revenue hours
top_revenue = hourly_revenue[
    hourly_revenue['order_price'] > avg_revenue_per_hour
]

plt.bar(
    x=top_revenue['order_hour'],
    height=top_revenue['order_price'],
    color='orange',
)
plt.title('Online Sales by Hours')
plt.xlabel('Hour')
plt.ylabel('Online Sales (AUD)')
plt.xticks(range(0, 24, 1))
plt.show()

The peak revenue hours that drive online sales above average are typically between **7 AM and 10 AM**, with the highest sales occurring at **8AM**.

### Revenue and Order Value per Customer

In [ ]:
df_revenue.head()

In [ ]:
cust_revenue = df_revenue[
    ['customer_id', 'order_year', 'order_month', 'order_id', 'order_price']
].sort_values(
    ['customer_id', 'order_year', 'order_month', 'order_id', 'order_price'],
    ascending=[True, True, True, True, True],
).reset_index(drop=True)

cust_revenue.head(10)

In [ ]:
# Create a DataFrame showing each customer's total spending
# over the entire business history in Descending order
total_spending_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .sum().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'total_spending'})
    .reset_index(drop=False)
)
# Calculate the average total spending
avg_total_spending = total_spending_per_cust['total_spending'].mean()

# Create a DataFrame showing the average spending per order by each
# customer in Descending order
spending_per_order_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .mean().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'spending_per_order'})
    .reset_index(drop=False)
)
# Calculate average customer spending per order
avg_spending_per_order = (
    spending_per_order_per_cust['spending_per_order'].mean()
)

# Create a DataFrame showing each customer's order frequency over 
# the entire business history in Descending order
order_frequency_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .count().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'order_freq'})
    .reset_index(drop=False)
)
# Calculate average order frequency
avg_order_frequency = (
    order_frequency_per_cust['order_freq'].mean()
)

In [ ]:
print(
    "The average customer total spending over their "
    "customer lifespan: ${}"
    .format(
        round(avg_total_spending, 2),
    )
)
total_spending_per_cust.head()

In [ ]:
print(
    "The average customer spending per order: ${}".format(
        round(avg_spending_per_order, 2)
    )
)
spending_per_order_per_cust.head()

In [ ]:
print(
    "The average order frequency: {}".format(
        round(avg_order_frequency)
    )
)
order_frequency_per_cust.head()

In [ ]:
top10_total_spending = total_spending_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_total_spending,
    x='total_spending',
    y='customer_id',
    orient='h',
    order=top10_total_spending['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Revenue-driving Customers')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Online Sales Revenue (AUD)')

plt.show()

In [ ]:
top10_order_freq = order_frequency_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_order_freq,
    x='order_freq',
    y='customer_id',
    orient='h',
    order=top10_order_freq['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Frequent Buyers')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Number of Orders')

plt.show()

In [ ]:
top10_spenders = top10_total_spending['customer_id']

top10_frequent_buyers = top10_order_freq['customer_id']

print(
    '{} out of the top 10 frequent buyers are also '
    'among the top 10 big spenders.'.format(
        len(
            [
                cust for cust in list(top10_frequent_buyers) 
                if cust in list(top10_spenders)
            ]
        )
    )
)

In [ ]:
top10_spending_per_order = spending_per_order_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_spending_per_order,
    x='spending_per_order',
    y='customer_id',
    orient='h',
    order=top10_spending_per_order['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Customers with Highest Average Spending per Order')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Average Spending Per Order (AUD)')

plt.show()

In [ ]:
top10_avg_spenders = top10_spending_per_order['customer_id']

print(
    '{} out of the top 10 customers with highest spending per order are also '
    'among the top 10 big spenders.'.format(
        len(
            [
                cust for cust in list(top10_avg_spenders) 
                if cust in list(top10_spenders)
            ]
        )
    )
)

In [ ]:
order_freq = order_frequency_per_cust[
    order_frequency_per_cust['customer_id'].isin(
        top10_spenders
    )
]
spending_per_order =  spending_per_order_per_cust[
    spending_per_order_per_cust['customer_id'].isin(
        top10_spenders
    )
]
spending_per_order.merge(
    order_freq,
    how='inner',
    on='customer_id',
)

#### Discuss Insights:

7 out of the top 10 highest spending customers are also among the top 10 repeat customers; however, none are in the top 10 for highest average spending per order. All of them have placed more orders than the average of 23, but have on average spent less per order than the mean of $17.32.

These observations suggest that customer spending may be more strongly correlated with the total number of orders they placed (order frequency) than with the average spending per order. To verify this, let's examine the **correlation coefficients** between these variables.

#### Determine Correlation Method

There are two common types of correlation coefficients we can calculate: **Pearson Correlation Coefficients** and **Spearman's Rank Correlation Coefficients**. Each method has specific assumptions:
* **Pearson Correlation Coefficients**:
    - The variables are normally distribtued
    - The relationship between the variables is **linear**
* **Spearman's Rank Correlation Coefficients**:
    - The data is not normally distributed
    - The relationship between variables is **monotonic** (consistently increases or decreases but not necessarily at a constant rate) but not necessarily linear
    - The data may contain outliers that could affect Pearson's correlation
    
Let's investigate the customer revenue data to determine which correlation method to use.

In [ ]:
df_corr = total_spending_per_cust.merge(
    order_frequency_per_cust,
    how='inner',
    on='customer_id',
).merge(
    spending_per_order_per_cust,
    how='inner',
    on='customer_id',
)

In [ ]:
# Plot kernel density distribution of customer spending
fig, ax = plt.subplots()
sns.kdeplot(data=df_corr, x='total_spending', ax=ax)
# Plot the mean of revenue by customers as a constant line
ax.plot(
    [df_corr['total_spending'].mean(), df_corr['total_spending'].mean()], 
    [0, 0.0018],
    linestyle='--',
    linewidth=0.8,
)
plt.xlabel('Total Spending by Customers (AUD)')
plt.show()

According to the Kernal Density Estimation plot above, the `total_spending` variable is NOT normally distributed (it is right-skewed) and contains outliers. Therefore, we should use the **Spearman's Rank Correlation Coefficients** to determine the correlations between `total_spending` and `order_freq`, and between `total_spending` and `spending_per_order`.

#### Calculate Spearman's Rank Correlation Coefficients

In [ ]:
# Compute Spearman's Rank Correlation Coefficients matrix 
# between total spending, order frequency and average spending
# per order
df_corr[
    ['total_spending', 'order_freq', 'spending_per_order']
].corr(method='spearman')

The **relatively high** correlation coefficient of **0.848067** between `total_spending` and `order_freq` indicates a **strong positive monotonic relationship** between these two variables. This suggests that as the order frequency increases, the total spending tend to increase as well. The high coefficient value signifies that customers who place orders more often generally contribute more to the total sales.

In contrast, the **relatively low** correlation coefficient (**0.395176**) between `total_spending` and `spending_per_order` indicates a **moderate positive monotonic relationship**. This means there is a weaker correlation between total spending and the average spending per order. While higher spending per order is somewhat associated with higher total sales, the relationship is not as strong or consistent as with the order frequency.

To determine whether the correlations between total spending and order frequency, and between total spending and average spending per order are statistically significant, we can perform **Hypothesis Test** on the Spearman's Rank Correlation Coefficients. This will help us assess the **level of confidence** we can place in the observed relationships.

#### Hypothesis Testing on Spearman's Rank Correlation Coefficients

**Hypotheses for `total_spending` vs. `order_freq`**

* Null Hypothesis ($H_0$): There is no monotonic relationship between `total_spending` and `order_freq` ($\rho = 0$)
* Alternative Hypothesis ($H_1$): There is a statistically significant monotonic relationship between `total_spending` and `order_freq` ($\rho\neq0$)

**Hypotheses for `total_spending` vs. `spending_per_order`**

* Null Hypothesis ($H_0$): There is no monotonic relationship between `total_spending` and `spending_per_order` ($\rho = 0$)
* Alternative Hypothesis ($H_1$): There is a statistically significant monotonic relationship between `total_spending` and `spending_per_order` ($\rho\neq0$)

**Significance Level ($\alpha$)**: 0.05

We will use the **Spearman's Rank Correlation Coefficients** and calculate the corresponding **p-value** to assess the **statistical significance** of the correlation. The p-value represents the probability of observing the data assuming that the null hypothesis is true. By convention, we set the **significance level** ($\alpha$) at **0.05**.

In [ ]:
total_spending = df_corr['total_spending']
order_freq = df_corr['order_freq']

# Calculate Spearman's Rank Correlation and p-value
# for total_spending vs. order_freq
corr_coefficient_ts_of, p_value_ts_of = spearmanr(
    total_spending, order_freq
)

print("Spearman's Rank correlation coefficient between "
      f'total_spending and order_freq: {corr_coefficient_ts_of}')
print(f'p-value: {p_value_ts_of}')

# Scatter plot for total_sales vs. order_count
plt.figure(figsize=(7, 5))
plt.scatter(
    order_freq, 
    total_spending, 
    edgecolors='#d3d3d3',
    s=60,
    linewidth=0.4,
    alpha=0.7,
)
# Adding a LOWESS line to represent a monotonic trend
sns.regplot(
    x=order_freq, 
    y=total_spending, 
    lowess=True, 
    scatter=False, 
    color='black',
    line_kws={
        'linewidth': 1.5,
        'linestyle': '--',
    },
)
plt.xlabel('Number of Orders')
plt.ylabel('Total Spending (AUD)')
plt.title('Total Spending vs. Order Frequency')
plt.show()

In [ ]:
total_spending = df_corr['total_spending']
spending_per_order = df_corr['spending_per_order']

# Calculate Spearman's Rank Correlation and p-value
# for total_spending vs. spending_per_order
corr_coefficient_ts_sp, p_value_ts_sp = spearmanr(
    total_spending, spending_per_order
)

print('Spearman correlation coefficient between '
      f'total_spending and spending_per_order: {corr_coefficient_ts_sp}')
print(f'p-value: {p_value_ts_sp}')

# Scatter plot for total_spending vs spending_per_order
plt.figure(figsize=(7, 5))
plt.scatter(
    spending_per_order, 
    total_spending, 
    edgecolors='#d3d3d3',
    s=60,
    linewidth=0.4,
    alpha=0.7,
)
# Adding a LOWESS line to represent a monotonic trend
sns.regplot(
    x=spending_per_order, 
    y=total_spending, 
    lowess=True, 
    scatter=False, 
    color='black',
    line_kws={
        'linewidth': 1.5,
        'linestyle': '--',
    },
)
plt.xlabel('Average Spending Per Order (AUD)')
plt.ylabel('Total Spending (AUD)')
plt.title('Total Customer Spending vs. Average Order Value')
plt.show()

**Results and Conclusion**:

* **Total Customer Spending vs. Order Frequency**:
    - Spearman's Rank Correlation Coefficient: **0.848**
        * Indicates a **strong positive monotonic relationship** between total customer spending and order frequency.
    - P-value: **0.0**
        * Since the p-value < 0.05, we **reject the null hypothesis**.
        * There is a **0% chance** that the observed correlation is due to randomness.
        * The strong positive correlation is **statistically significant**.
<br><br>
* **Total Customer Spending vs. Average Spending per Order**:
    - Spearman's Rank Correlation Coefficient: **0.395**.
        * Indicates a **moderate positive monotonic relationship** between total customer spending and average spending per order.
    - P-value: **1.68e-59**
        * Since the p-value < 0.05, we **reject the null hypothesis**.
        * There is an **extremely low chance** (effectively 0%) that the observed correlation is due to randomness.
        * The moderate positive correlation is **statistically significant**.

#### Summary

Due to the non-normal distribution of the variables, we used **Spearman's Rank Correlation** to assess the direction, strength, and statistical significance of the **monotonic relationships** between total spending and the order frequency, as well as between total spending and average spending per order. Both correlations are **positive** and **statistically significant** at the 0.05 significance level, reinforcing the validity of observed relationships. We are now **confident** that:

* There is a **strong positive monotonic relationship** between customers' total spending and their order frequency. As customers place orders more frequently, their total spending tend to increase accordingly.

* There is a **moderate positive monotonic relationship** between customers' total spending and their average spending per order. Higher average spending per order is somewhat associated with increased total spending, but this relationship is not as strong as with the order frequency.

This analysis suggests that **repeat customers contribute more significantly to total sales** than customers who make high-value purchases per order. Therefore, implementing business strategies (e.g. loyalty program, repeat purchase incentives) aimed at **enhancing customer retention**, especially among the most loyal customers, and **encouraging repeat purchases**, is likely to be more effecitve in boosting total online sales than efforts focused on increasing their average spending per order.

<hr>

## The Customer Churn Risk of Products

Since the café updates its menu annually every July, we will analyse orders placed from July 2022 onward to evaluate each product's recent impact on customer churn and retention.

### Determining whether each order placed since July 2022 was followed by a repeat purchase

For each customer's orders, if the number of days between the current order and the next one does NOT exceed the churn threshold, the next order is considered a **repeat purchase**. Otherwise, it is classified as a **new purchase**.

In [ ]:
# Get the latest order time in the dataset
latest_time = df['order_time'].max()

# Define the start date for filtering orders from 1st Jul 2022
start_date = datetime(2022, 7, 1)

# Filter the orders placed since Jul 2022 and select relevant columns
# and sort the DataFrame
df_orders = df[
    df['order_time'] >= start_date
][
    ['customer_id', 'order_id', 'order_time', 'churn_threshold']
].sort_values(
    ['customer_id', 'order_time', 'order_id'],
    ascending=[True, True, True],
).drop_duplicates().reset_index(drop=True)

# Create a new column 'next_order_time' that holds the next 
# order time for each customer
df_orders['next_order_time'] = (
    df_orders.groupby(['customer_id'])['order_time'].shift(-1)
)

# Calculate the interval in days between the current order and the 
# next order
df_orders['interval_to_next'] = df_orders.apply(
    lambda x: (x['next_order_time'] - x['order_time']).days,
    axis=1,
)

# Define a function to determine whether the current purchase has a 
# succeeding repeat purchase based on the churn threshold
def determine_next_repeat_purchase(x):
    # If no next order exists, calculate interval in days from the current 
    # order to the latest time
    if pd.isna(x['next_order_time']):
        interval_to_latest = (latest_time - x['order_time']).days
        # If this interval is less than the churn threshold, mark it as 'Unknown'
        if interval_to_latest < int(x['churn_threshold']):
            return 'Unknown'
    # If the interval to the next order is less than the churn threshold, 
    # mark it as a repeat purchase
    if x['interval_to_next'] < x['churn_threshold']:
        return True
    else:
        return False

# Apply the function to determine if the current purchase has a succeeding
# repeat purchase
df_orders['next_repeat_purchase'] = df_orders.apply(
    determine_next_repeat_purchase,
    axis=1,
)

# Filter out orders where the 'next_repeat_purchase' status is 'Unknown'
df_orders = df_orders[
    df_orders['next_repeat_purchase'] != 'Unknown'
]

In [ ]:
df_orders[
    df_orders['customer_id'] == 9
][
    ['order_id', 'order_time', 'interval_to_next', 'churn_threshold', 'next_repeat_purchase']
].head()

In [ ]:
df_orders[
    df_orders['customer_id'] == 9
]['next_repeat_purchase'].value_counts()

In [ ]:
# Select relevant columns for item analysis and sort the DataFrame
df_items = df[
    ['customer_id', 'order_id', 'order_time', 'item_tracking_id', 
     'item', 'quantity']
].sort_values(
    ['customer_id', 'order_time', 'order_id', 'item_tracking_id'],
    ascending=[True, True, True, True],
).drop_duplicates().reset_index(drop=True)

# Append the `next_repeat_purchase` column to the items data
df_items = df_items.merge(
    df_orders[['order_id', 'next_repeat_purchase']],
    how='right',
    on='order_id',
)

In [ ]:
df_items[
    df_items['order_id'].isin([33358, 32760, 34894, 38779])
]

### Applying Bayes' Theorem to calculate the customer retention probability after buying each particular item

To investigate whether buying a specific product would result in customer retention (or a repeat purchase), we can use **Bayes' Theorem** to determine the probability that a customer will make a repeat purchase in the near future after buying the specific item.

The **Bayes' Theorem** allows us to update our probability estimates for an event (a customer making a subsequent repeat purchase) based on new evidence (having purchased a specific item in the current order). The theorem in our case is methematically expressed as:

$$ P\text{(repeat|product)} = \frac{P\text{(product|repeat)} \times P\text{(repeat)}}{P\text{(product)}} $$

Where:
* **$P\text{(repeat|product)}$**: **Posterior Probability** - the probability that a customer will make a subsequent repeat purchase given that they have purchased the specific item in the current order.
* **$P\text{(product|repeat)}$**: **Likelihood** - the probability of purchasing the specific item in the current order given that the next order is a repeat purchase.
* **$P\text{(repeat)}$**: **Prior Probability** - the initial probability of the next order being a repeat purchase without considering the current purchase.
* **$P\text{(product)}$**: **Marginal Probability** - the probability of purchasing the specific item in the current order.

The reasons we apply **Bayes' Theorem** include:
1) Conditional Probability Estimation: The theorem is designed to compute conditional probabilities, which is exactly what's needed in this scenario: determining the customer retention probability **$P$(repeat | product)**.
2) Incorporating New Evidence: The purchase of a specific item serves as new evidence that can influence the likelihood of a subsequent repeat purchase. Bayes' Theorem provides a structured way to incorporate this evidence into the probability assessment.
3) Flexibility with Prior Information: It allows the integration of prior knowledge or historical data ($P$(repeat)) with the new evidence ($P$(product | repeat)) to update the probability assessment.
4) Assumption of Independence: Bayes' Theorem assumes that the evidence (having purchased a specific item in the current order) is conditionally independent of other factors given the hypothesis of next order being a repeat purchase.

In [ ]:
# Count the number of orders for each item that are followed by a 
# repeat purchase
df_repeat_item_count = (
    df_items[
        df_items['next_repeat_purchase']
    ][
        ['item', 'order_id']
    ].drop_duplicates().groupby('item').count()
    .reset_index(drop=False)
    .rename(columns={'order_id': 'item_order_count_next_repeat'})
)

# Count the total number of orders for each item
df_total_item_count = (
    df_items[
        ['item', 'order_id']
    ].drop_duplicates().groupby('item').count()
    .reset_index(drop=False)
    .rename(columns={'order_id': 'item_order_count'})
)

# Merge the total item purchase counts with the repeat purchase
# counts
df_item_count = df_total_item_count.merge(
    df_repeat_item_count,
    how='inner',
    on='item',
)

df_item_count.head()

To interpret the data in the DataFrame, Let's take the **(Entree) Pasta** item as an example. This item has been purchased in **3 orders** (`item_order_count`), **2 of which** have subsequent repeat purchases (`item_order_count_next_repeat`).

#### Calculating the estimated probability $P(repeat | product)$

$$ P\text{(repeat|product)} = \frac{P\text{(product|repeat)} \times P\text{(repeat)}}{P\text{(product)}} $$

We currently have the following data available:
* Total number of orders ($N_{\text{total}}$)
* Total number of orders for each item ($N_{\text{item_total}}$)
* Number of orders for each item that are followed by a repeat purchase ($N_{\text{item_repeat}}$)
* Total number of orders with a subsequent repeat purchase ($N_{\text{repeat}}$)

The **Prior Probability** $P\text{(repeat)}$, **Likelihood** $P\text{(product|repeat)}$, and **Marginal Probability** $P\text{(product)}$ for the Bayes' Theorem exression can be calculated as below shows:

$$P\text{(repeat)} = \frac {N_{\text{repeat}}} {N_{\text{total}}}$$

$$P\text{(product)} = \frac {N_{\text{item_total}}} {N_{\text{total}}}$$

$$P\text{(product|repeat)} = \frac {N_{\text{item_repeat}}} {N_{\text{repeat}}}$$

Plugging them into the **Bayes' Theorem exression**, we can get:

$$ P\text{(repeat|product)} = \frac{P\text{(product|repeat)} \times P\text{(repeat)}}{P\text{(product)}} 
    = \frac {(\frac {N_{\text{item_repeat}}} {N_{\text{repeat}}}) \times (\frac {N_{\text{repeat}}} {N_{\text{total}}})} {(\frac {N_{\text{item_total}}} {N_{\text{total}}})}
    = \frac {N_{\text{item_repeat}}}{N_{\text{item_total}}}
$$

Therefore, **the estimated probability $P\text{(repeat|product)}$ that a customer will make a subsequent repeat purchase given that they have purchased a specific item in the current order** is equal to the **ratio** of **the number of orders for that item that are followed by a repeat purchase** $N_{\text{item_repeat}}$ to **the total number of purchases of that item** $N_{\text{item_total}}$.

### Smoothing Probability Estimates

The calculated probability $P\text{(repeat|product)}$ could end up being exact 0 or 1 if none of the orders for a particular item has ever been followed by a repeat purchase, or if every order of that item has a subsequent repeat purchase. This can be misleading since the lack of historical data doesn't necessarily mean a subsequent repeat purchase event is impossible or guaranteed. 

To avoid $P\text{(repeat|product)}$ being exactly 0 or 1, we can introduce **Smoothing** in the probability calculations. This approach can help prevent overconfidence in such cases, especially with limited sample sizes, providing more robust and realistic probability estimates.

Having introduced **Smoothing**, the Bayes' Theorem formula can now be expressed as:

$$ P\text{(repeat|product)} = \frac {N_{\text{item_repeat}} + \alpha}{N_{\text{item_total}} + 2\alpha} $$

Where:
* $\alpha$: Smoothing factor. Since we are dealing with binary outcomes (a customer will or will not make a subsequent repeat purchase), we need to add twice the smoothing factor to the denominator. This ensures that the total probability across both outcomes sums to 1, maintaining the balance between the two outcomes in the probability calculations.

While a smoothing factor of 1 is commonly used, it may be too large in our case, as the sample size (the number of purchases of a particular item)  can be very small. Using a smaller smoothing factor, such as **0.005**, would introduce a smaller adjustment compared to 1, which better accounts for very small sample sizes, while still preventing probabilities from being exactly 0 or 1.

In [ ]:
# Set the smoothing factor at 0.005
smoothing = 0.005

# Calculate the estimated customer retention probability given that
# a particular item was purchased in the current order
df_item_count['P_repeat_product'] = (
    (df_item_count['item_order_count_next_repeat'] + smoothing)
    / (df_item_count['item_order_count'] + 2 * smoothing)
)

df_item_count.head()

In [ ]:
print(df_item_count['P_repeat_product'].min())
print(df_item_count['P_repeat_product'].max())

The estimated probabilities now range from 0.2506 to 0.9997.

### Classifying Churn Risk of Products based on their Confidence Interval for Binomial Proportion

Given that the sample size (the number of purchases of a particular item) can be very **limited**, their individual probability estimates may be **unreliable** for the following reasons:
1) When the sample size is small, the **randomness** in the data becomes more pronounced. This makes it more likely that the estimated customer retention probabililty for an item is either overestimated or underestimated due to **chance occurrences** rather than a true reflection of customer behaviour for that item.
2) Small sample sizes make estimates **more susceptible** to **outliers** or **extreme values**. For example, if a product has been purchased in only a few orders, but none of them were followed by a repeat purchase, it may incorrectly suggest that buying the item would almost never lead to customer retention. This overconfident conclusion is based on insufficient data and is unlikely to generalize to the customers' true behaviour for that item. 
3) With a small sample size, data can become **highly sensitive to small changes** - even one or two additional order observations can markedly change the estimated probability. For example, if only 9 orders with no subsequent repeat purchases have been placed for a particular item, and a single repeat purchase occurs, the customer retention probability resulting from buying the item might jump from 0% to 10%. In contrast, with a larger sample size (say 1000 orders), adding one repeat purchase has much less impact, resulting in a more stable estimate.
4) The **Law of Large Numbers** states that as the sample size increases, the sample mean (or probability mean) will converge toward the true population mean. When the sample size is small, the estimated probability is more likely to **deviate from the true probability**, leading to a biased estimate of an item's true customer retention probability.

#### Calculating Wilson Score Interval

**Confidence Interval for Binomial Proportion** provides a range in which the true probability is likely to lie. With **smaller sample sizes**, the **confidence interval** becomes **wider**, reflecting **greater uncertainty** in the estimate. We can use the confidence interval to **quantify the reliability** of our estimates.

In our case, we will calculate **Wilson Score Interval** to assess the uncertainty in our estimated probabilities, as it doesn't depend on the normal approximation, making it more appropriate for small sample sizes. The items will then be categorised into different **churn risk levels** based on the **lower bound** and **upper bound** of the corresponding interval.

The lower and upper confidence bounds from the **Wilson Score Interval** are calculated as:

$$
\text{Lower Bound} = \frac{ \hat{p} + \frac{Z^2}{2n} - Z \sqrt{ \frac{ \hat{p}(1 - \hat{p}) }{n} + \frac{Z^2}{4n^2} } }{1 + \frac{Z^2}{n}}
$$

$$
\text{Upper Bound} = \frac{ \hat{p} + \frac{Z^2}{2n} + Z \sqrt{ \frac{ \hat{p}(1 - \hat{p}) }{n} + \frac{Z^2}{4n^2} } }{1 + \frac{Z^2}{n}}
$$


Where:
* $\hat{p} = \frac {x} {n}$ : The estimated probability that a customer will make a subsequent repeat purchase given that they have purchased a specific item in the current order.
* $x$: The number of orders for the item that are followed by a repeat purchase.
* $n$: The total number of orders for the item.
* $Z$: The Z-score for the desired confidence level (e.g. 1.96 for 95% confidence level).

In [ ]:
# Caculate the lower bound and upper bound of the Wilson
# Score Interval for each item
ci_lower, ci_upper = proportion_confint(
    count=df_item_count['item_order_count_next_repeat'],  # The number of orders with a subsequent repeat purchase for each item
    nobs=df_item_count['item_order_count'],  # The total number of orders for each item
    alpha=0.05,   # Significance level = 1 - confidence level. An alpha of 0.05 means the desired confidence level is 95%
    method='wilson',  # Wilson Score Interval method to use for confidence interval
)

df_item_count['ci_lower'] = ci_lower
df_item_count['ci_upper'] = ci_upper

In [ ]:
df_item_count.head()

#### Classifying Items based on Confidence Interval

Having calculated the Wilson Score Interval, we can categorise items into different churn risk levels based on the **lower bound** and **upper bound** of the interval.

In our analysis, we will use the **median** of the estimated probabilities across all items as a **threshold**, comparing it to the lower and upper bounds of each item's confidence interval to classify the **churn risk** of each item as **high**, **moderate**, and **low**, according to the following criteria:

* **Low Churn Risk Items**: Items whose **lower bound** of the confidence interval is **above** the threshold are classified as **High Retention Items**, or **Low Churn Risk Items**. Even at their lowest possible customer retention probability, the orders for these items have a relatively high chance of being followed by a repeat purchase, meaning that they present a **low risk** of contributing to customer churn.
* **High Churn Risk Items**: Items whose **upper bound** of the confidence interval falls **below** the threshold are classified as **Low Retention Items**, or **High Churn Risk Items**. Even at their highest possible customer retention probability, the orders for these items have a relatively low chance of being followed by a repeat purchase. This suggests that these items pose a **high risk** of contributing to customer churn.
* **Moderate Churn Risk Items**: Items with a lower bound below the threshold but an upper bound above the threshold are classified as **Moderate Retention Items**, or **Moderate Churn Risk Items**. This indicates that they carry a **moderate risk** of contributing to customer churn.

In [ ]:
# Calculate the median of the estimated probabilities across all items
median_prob = df_item_count['P_repeat_product'].median()

print(median_prob)

In [ ]:
# Classify each item's churn risk by comparing the lower 
# and upper bounds of their confidence interval against
# the median of probability estimates
def classify_churn_risk(x):
    if x['ci_lower'] > median_prob:
        return 'Low'
    elif x['ci_upper'] < median_prob:
        return 'High'
    else:
        return 'Moderate'

df_item_count['churn_risk_level'] = df_item_count.apply(
    classify_churn_risk, axis=1,
)

In [ ]:
df_item_count.head()

In [ ]:
# View the number of items in each churn risk category
df_item_count['churn_risk_level'].value_counts()

**10** products are classified as **High Churn Risk Item** (or **Low Retention Item**), **19** as **Low Churn Risk Item** (or **High Retention Item**), and **45** as **Moderate Churn Risk Item** (or **Moderate Retention Item**).

In [ ]:
# Create a new dataframe for items with high churn risk
items_high_churn_risk = df_item_count[
    df_item_count['churn_risk_level'] == 'High'
].sort_values('P_repeat_product', ascending=True)

# Create a new dataframe for items with high retention probability
items_high_retention = df_item_count[
    df_item_count['churn_risk_level'] == 'Low'
].sort_values('P_repeat_product', ascending=False)

In [ ]:
# Plot customer rentention probability over high retention items
plt.figure(figsize=(8, 6))

# Remove the top and right borders of the chart
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Horizontal bar chart for customer rentention probability 
# over high retention items
sns.barplot(
    y=items_high_retention['item'], 
    x=items_high_retention['P_repeat_product'], 
    color='#9d6346',
    order=items_high_retention['item'],
)

for i in range(len(items_high_retention)):
    # Add Gantt-like bars for confidence intervals
    plt.hlines(
        y=i, 
        xmin=items_high_retention['ci_lower'].iloc[i], 
        xmax=items_high_retention['ci_upper'].iloc[i], 
        color='black', 
        linewidth=2,
    )
    
    # Add a dot at lower bound of each confidence interval
    plt.scatter(
        y=i, 
        x=items_high_retention['ci_lower'].iloc[i], 
        color='black',
        s=10,
    )
    
    # Add a dot at upper bound of each confidence interval
    plt.scatter(
        y=i,
        x=items_high_retention['ci_upper'].iloc[i],
        color='black',
        s=10,
    )

# Add a constant vertical line of the median of estimated probabilities
# at x-axis
plt.axvline(
    x=median_prob, 
    color='#a9a9a9', 
    linestyle='--', 
    label='Median Customer Retention Probability',
)
    
# Add title and labels
plt.title(
    '\nItems with High Customer Retention Probability \nBetween Jul 2022 and Mar 2024\n', 
    fontsize=14,
)
plt.legend()
plt.xticks(np.linspace(0, 1, 11, endpoint=True))
plt.xlabel('Customer Retention Probability', fontsize=12)
plt.ylabel('')
plt.show()

# Display the dataframe for items with high retention probability
items_high_retention

In [ ]:
# Plot customer rentention probability over high churn risk items
plt.figure(figsize=(8, 6))

# Remove the top and right borders of the chart
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Horizontal bar chart for customer rentention probability 
# over high churn risk items
sns.barplot(
    y=items_high_churn_risk['item'], 
    x=items_high_churn_risk['P_repeat_product'], 
    color='#f4d03f',
    order=items_high_churn_risk['item'],
)

for i in range(len(items_high_churn_risk)):
    # Add Gantt-like bars for confidence intervals
    plt.hlines(
        y=i, 
        xmin=items_high_churn_risk['ci_lower'].iloc[i], 
        xmax=items_high_churn_risk['ci_upper'].iloc[i], 
        color='black', 
        linewidth=2,
    )
    
    # Add a dot at lower bound of each confidence interval
    plt.scatter(
        y=i, 
        x=items_high_churn_risk['ci_lower'].iloc[i], 
        color='black',
        s=10,
    )
    
    # Add a dot at upper bound of each confidence interval
    plt.scatter(
        y=i,
        x=items_high_churn_risk['ci_upper'].iloc[i],
        color='black',
        s=10,
    )

# Add a constant vertical line of the median of estimated probabilities
# at x-axis
plt.axvline(
    x=median_prob, 
    color='#a9a9a9', 
    linestyle='--', 
    label='Median Customer Retention Probability',
)

plt.title(
    '\nItems with High Customer Churn Risk\nBetween Jul 2022 and Mar 2024\n', 
    fontsize=14,
)
plt.legend()
plt.xticks(np.linspace(0, 1, 11, endpoint=True))
plt.xlabel('Customer Retention Probability', fontsize=12)
plt.ylabel('')
plt.show()

# Display the dataframe for items with high churn risk
items_high_churn_risk

We may consider **removing those high churn risk items from the cafe's menu** to enhance customer retention.

## Strategy Analysis: Removing High Churn Risk Items from Café Menu

We will start with calculating the **churn rate** for **existing customers**, **new customers**, and **high-spending customers**, respetively. Then 

### Calculating Customer Churns for Different Groups of Customers

#### Computing churn status for each customer at both the start and end of the period

In [ ]:
# Load the monthly churn data into a pandas DataFrame
df_monthly_churn = pd.read_parquet(
    './cafe_processed_files/cafe_customer_churn_mx.parquet'
)

In [ ]:
def compute_start_end_churn(df_churn):
    # Get the customer churn data as of the latest month period,
    # which is Mar 2024
    latest_month = df_churn['as_of'].max()

    end_period_churn = df_churn[
        df_churn['as_of'] == latest_month
    ][['customer_id', 'churn', 'latest_order_time']].rename(
        columns={'churn': 'end_churn'}
    )
    
    # Get the customer churn data as of the 1st Jul 2022
    start_period_churn = df_churn[
        df_churn['as_of'] == datetime(2022, 7, 1)
    ][['customer_id', 'churn']].rename(
        columns={'churn': 'start_churn'}
    )
    
    # Merge start and end churn statuses into a single dataframe
    cust_start_end_churn = end_period_churn.merge(
        start_period_churn,
        how='left',
        on='customer_id',
    )[
        ['customer_id', 'start_churn', 'end_churn', 'latest_order_time']
    ]
    
    return cust_start_end_churn

In [ ]:
# Compute the churn status for each customer at both the start and 
# end of the period between Jul 2022 and Mar 2024
cust_start_end_churn = compute_start_end_churn(df_monthly_churn)
cust_start_end_churn.head()

#### Calculating the threshold for identifying high-spending and high-frequency customers

We will use the **80th Percentile (Top 20%)** of total spending and order frequency **among repeat customers** (those who made more than one purchase) as thresholds to identify **high-spending** and **high-frequency** customers, respectively.

This approach is inspired by the **Pareto Principle**, also known as the **80/20 rule**, which suggests that roughly 80% of outcomes (in our case, the total sales and total number of orders) are often driven by 20% contributors (i.e. top customers). Focusing on the **top 20% of customers in highest total spending** enables us to effectively target the **most valuable customers** that drive the majority of business online sales, while analyzing **top 20% of customers in highest order frequency** allows us to better understand our **most loyal customers**.

Additionally, we apply this analysis specifically to **repeat customers** - those who made more than one purchase. One-time purchasers could skew the percentiles downward, resulting in **unrealistically low thresholds** that do not accurately reflect the behaviour of loyal customers. By excluding these buyers, we ensure that the thresholds are based on the spending and purchase patterns of ongoing customers, leading to more accurate and meaningful segmentation.

In [ ]:
def compute_spending_freq_thresholds(df):
    # Create a new dataframe for each customer's unique orders
    df_orders = df[
        ['customer_id', 'order_id', 'order_time', 'order_price']
    ].sort_values(
        ['customer_id', 'order_time', 'order_id'],
        ascending=[True, True, True],
    ).drop_duplicates().reset_index(drop=True)

    # Calculate the total spending and order frequency for each customer
    df_spending_freq = df_orders.groupby('customer_id').agg(
        {
            'order_price': 'sum',
            'order_id': 'count',
        }
    ).rename(
        columns={'order_price': 'total_spending', 'order_id': 'order_freq'}
    ).reset_index(drop=False)
    
    # Calculate 80th percentile of total spending as the threshold 
    # for identifying high-spending customers among repeat purchasers
    total_spending_threshold = df_spending_freq[
        df_spending_freq['order_freq'] > 1
    ]['total_spending'].quantile(0.8)
    
    # Calculate 80th percentile of order frequency as the threshold  
    # for identifying high-frequency customers among repeat purchasers
    order_freq_threshold = df_spending_freq[
        df_spending_freq['order_freq'] > 1
    ]['order_freq'].quantile(0.8)

    print('Threshold for identifying high-spending customers:', total_spending_threshold)
    print('Threshold for identifying high-frequency customers:', order_freq_threshold)
    
    return total_spending_threshold, order_freq_threshold, df_spending_freq

In [ ]:
# Compute the thresholds for high-spending and high-frequency customers respectively
total_spending_threshold, order_freq_threshold, df_spending_freq = (
    compute_spending_freq_thresholds(df)
)

In [ ]:
df_spending_freq.head()

In [ ]:
def plot_pareto_chart(df_spending_freq, target_col, threshold):
    # Keep repeat customers only and sort by target metric in descending
    # order
    df_repeat_cust = df_spending_freq[
        df_spending_freq['order_freq'] > 1
    ].sort_values(
        target_col, ascending=False
    ).reset_index(drop=True)

    # Calculate the proportion of high-spending (or high-frequency customers) customers,   
    # which is the number of repeat customers whose total spending (or order frequency)  
    # exceeds the threshold (80th percentile) divided by the total number of repeat 
    # customers
    high_cust_prop_repeat = len(df_repeat_cust[
        df_repeat_cust[target_col] > threshold
    ]) / len(df_repeat_cust)

    # Calculate the cumulative total spending (or cumulative total number of orders)  
    # for the top 20% (80th percentile) of customers
    cum_total_80pct_repeat = (
        df_repeat_cust[target_col].cumsum()[
            int(high_cust_prop_repeat * len(df_repeat_cust))
        ]
    )

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), sharey=True)

    # Plot the cumulative total of target metric against the percentage of top customers
    axes[0].plot(
        np.linspace(0, 1, len(df_repeat_cust), endpoint=True),
        df_repeat_cust[target_col].cumsum(),
        c='#9b442a',
    )

    # Add a vertical dashed line at the proportion of high-spending 
    # (or high-frequency customers) customers (80th percentile)
    axes[0].plot(
        [high_cust_prop_repeat, high_cust_prop_repeat],
        [0, df_repeat_cust[target_col].cumsum().max()],
        linestyle='--',
        color='#a9a9a9',
    )

    # Add a horizontal dashed line indicating the cumulative total spending (or  
    # cumulative total number of orders) at the 80th percentile
    axes[0].plot(
        [0, 1],
        [
            cum_total_80pct_repeat,
            cum_total_80pct_repeat,
        ],
        linestyle='--',
        color='#a9a9a9',
    )

    # Add label to the vertical dashed line
    axes[0].text(
        high_cust_prop_repeat + 0.02,  # Slightly to the right of the vertical line
        0,  # At the bottom
        str(round(high_cust_prop_repeat * 100)) + '%',  # Label
        verticalalignment='bottom',
        fontsize=12,
        color='black',
    )

    # Add label to the horizontal dashed line
    axes[0].text(
        1,  # Far right of the plot
        cum_total_80pct_repeat + (0.02 * cum_total_80pct_repeat),  # Slightly above the line
        str(
            round(
                cum_total_80pct_repeat / df_repeat_cust[target_col].sum() * 100
            )
        ) + '%',  # Label
        horizontalalignment='right',
        fontsize=12,
        color='black',
    )
    
    # Keep repeat customers only and sort by target metric in descending
    # order
    df_cust = df_spending_freq.sort_values(
        target_col, ascending=False
    ).reset_index(drop=True)

    # Calculate the proportion of high-spending (or high-frequency customers) customers,   
    # which is the number of repeat customers whose total spending (or order frequency)  
    # exceeds the threshold (80th percentile) divided by the total number of repeat 
    # customers
    high_cust_prop = len(df_cust[
        df_cust[target_col] > threshold
    ]) / len(df_cust)

    # Calculate the cumulative total spending (or cumulative total number of orders)  
    # for the top 20% (80th percentile) of customers
    cum_total_80pct = (
        df_cust[target_col].cumsum()[
            int(high_cust_prop * len(df_cust))
        ]
    )

    # Plot the cumulative total of target metric against the percentage of top customers
    axes[1].plot(
        np.linspace(0, 1, len(df_cust), endpoint=True),
        df_cust[target_col].cumsum(),
        c='#9b442a',
    )

    # Add a vertical dashed line at the proportion of high-spending 
    # (or high-frequency customers) customers (80th percentile)
    axes[1].plot(
        [high_cust_prop, high_cust_prop],
        [0, df_cust[target_col].cumsum().max()],
        linestyle='--',
        color='#a9a9a9',
    )

    # Add a horizontal dashed line indicating the cumulative total spending (or  
    # cumulative total number of orders) at the 80th percentile
    axes[1].plot(
        [0, 1],
        [
            cum_total_80pct,
            cum_total_80pct,
        ],
        linestyle='--',
        color='#a9a9a9',
    )

    # Add label to the vertical dashed line
    axes[1].text(
        high_cust_prop + 0.02,  # Slightly to the right of the vertical line
        0,  # At the bottom
        str(round(high_cust_prop * 100)) + '%',  # Label
        verticalalignment='bottom',
        fontsize=12,
        color='black',
    )

    # Add label to the horizontal dashed line
    axes[1].text(
        1,  # Far right of the plot
        cum_total_80pct + (0.02 * cum_total_80pct),  # Slightly above the line
        str(
            round(
                cum_total_80pct / df_cust[target_col].sum() * 100
            )
        ) + '%',  # Label
        horizontalalignment='right',
        fontsize=12,
        color='black',
    )

    if target_col == 'total_spending':
        title_repeat = 'Cumulative Customer Spending\nby Top High-Spending Repeat Customers\n'
        title = 'Cumulative Customer Spending\nby Top High-Spending Customers\n'
        ylabel = 'Running Total of Customer Spending'
    if target_col == 'order_freq':
        title_repeat = 'Cumulative Number of Orders\nby Top Loyal Customers\n'
        title = 'Cumulative Number of Orders\nby Top Customers in Order Frequency\n'
        ylabel = 'Running Total of Number of Orders'

    # Add title and labels for the chart
    axes[0].set_title(title_repeat, size=14)
    axes[1].set_title(title, size=14)
    axes[0].set_ylabel(ylabel)
    axes[0].set_xlabel('Percentage of Top Repeat Customers')
    axes[1].set_xlabel('Percentage of Top Repeat Customers')
    
    plt.show()

In [ ]:
plot_pareto_chart(df_spending_freq, 'total_spending', total_spending_threshold)

The **top 20% (80th percentile)** of repeat customers in highest total spending accounted for **80%** of total online sales from repeat customers, and **77%** of total online sales across the entire customer base.

In [ ]:
plot_pareto_chart(df_spending_freq, 'order_freq', order_freq_threshold)

The **top 20% (80th percentile)** of loyal customers, based on order frequency, contributed **84%** of the total orders placed by repeat customers, and **82%** of the total orders across the entire customer base.

#### Segmenting customers into Existing, New, High-Spending, and High-Frequency groups

In [ ]:
def segment_customers(cust_start_end_churn, df_spending_freq):
    # Determine whether each customer is an existing customer
    # as of 1st of Jul 2022
    cust_start_end_churn['existing'] = cust_start_end_churn.apply(
        lambda x: True if (x['start_churn'] == False)
        and (x['latest_order_time'] >= datetime(2022, 7, 1))
        else False,
        axis=1,
    )

    # Determine whether each customer is a new customer during
    # the period between Jul 2022 and Mar 2024
    cust_start_end_churn['new'] = cust_start_end_churn.apply(
        lambda x: True if (x['start_churn'] != False)
        and (x['latest_order_time'] >= datetime(2022, 7, 1))
        else False,
        axis=1,
    )

    # Determine whether each customer is a high-spending customer
    df_spending_freq['high-spending'] = df_spending_freq.apply(
        lambda x: True if x['total_spending'] > total_spending_threshold
        else False,
        axis=1,
    )
    
    # Determine whether each customer is a high-frequency customer
    df_spending_freq['high-frequency'] = df_spending_freq.apply(
        lambda x: True if x['order_freq'] > order_freq_threshold
        else False,
        axis=1,
    )
    
    # Merge the group labeling into a single dataframe
    df_cust_segmented = cust_start_end_churn.drop(
        'latest_order_time', axis=1
    ).merge(
        df_spending_freq.drop(
            ['total_spending', 'order_freq'], axis=1
        ),
        how='right',
        on='customer_id',
    )
    
    return df_cust_segmented

In [ ]:
df_cust_segmented = segment_customers(cust_start_end_churn, df_spending_freq)
df_cust_segmented.head()

#### Calculating churn rates and number of churned customers across different segments

The **Churn Rate** for each customer segment can be calculated as:

$$ \text{Churn Rate} = \frac{\text{Total Number of Churned Customers}} {\text{Total Number of Customers}} $$

In [ ]:
def compute_churns(df_cust_segmented):
    """A function to calculate churn rates and number of churned customers 
    across different customer segments.
    """
    # Filter for existing customers
    existing_customers = df_cust_segmented[
        df_cust_segmented['existing']
    ]
    
    # Filter existing customers who churned by the end of
    # the period
    existing_customers_churned = existing_customers[
        existing_customers['end_churn']
    ]

    # Filter for existing high-spending customers
    existing_high_spending_customers = existing_customers[
        existing_customers['high-spending']
    ]

    # Filter existing high-spending customers who churned by the
    # end of the period
    existing_high_spending_customers_churned = (
        existing_high_spending_customers[
            existing_high_spending_customers['end_churn']
        ]
    )
    
    # Filter for existing high-frequency customers
    existing_high_freq_customers = existing_customers[
        existing_customers['high-frequency']
    ]

    # Filter existing high-frequency customers who churned by the
    # end of the period
    existing_high_freq_customers_churned = (
        existing_high_freq_customers[
            existing_high_freq_customers['end_churn']
        ]
    )

    # Filter for new customers acquired during the period
    new_customers = df_cust_segmented[
        df_cust_segmented['new']
    ]

    # Filter new customers who churned by the end of the period
    new_customers_churned = new_customers[
        new_customers['end_churn']
    ]

    # Filter for new high-spending customers
    new_high_spending_customers = new_customers[
        new_customers['high-spending']
    ]

    # Filter new high-spending customers who churned by the end of
    # the period
    new_high_spending_customers_churned = (
        new_high_spending_customers[
            new_high_spending_customers['end_churn']
        ]
    )
    
    # Filter for new high-frequency customers
    new_high_freq_customers = new_customers[
        new_customers['high-frequency']
    ]

    # Filter new high-frequency customers who churned by the end of
    # the period
    new_high_freq_customers_churned = (
        new_high_freq_customers[
            new_high_freq_customers['end_churn']
        ]
    )

    # Filter for all active customers during period (including both 
    # existing and new customers)
    overall_customers = df_cust_segmented[
        df_cust_segmented['existing']
        | df_cust_segmented['new']
    ]

    # Filter the active customers who churned by the end of the period
    overall_customers_churned = overall_customers[
        overall_customers['end_churn']
    ]

    # Filter for high-spending active customers
    high_spending_customers = overall_customers[
        overall_customers['high-spending']
    ]

    # Filter high-spending customers who churned by the end of the period
    high_spending_customers_churned = high_spending_customers[
        high_spending_customers['end_churn']
    ]
    
    # Filter for high-frequency active customers
    high_freq_customers = overall_customers[
        overall_customers['high-frequency']
    ]

    # Filter high-frequency customers who churned by the end of the period
    high_freq_customers_churned = high_freq_customers[
        high_freq_customers['end_churn']
    ]

    # Compute the churn rate for all active customers
    churn_rate_overall = len(overall_customers_churned) / len(overall_customers)

    # Compute the churn rate for existing customers
    churn_rate_existing = len(existing_customers_churned) / len(existing_customers)

    # Compute the churn rate for new customers
    churn_rate_new = len(new_customers_churned) / len(new_customers)

    # Compute the churn rate for high-spending customers
    churn_rate_high_spending = (
        len(high_spending_customers_churned) / len(high_spending_customers)
    )
    
    # Compute the churn rate for high-frequency customers
    churn_rate_high_freq = (
        len(high_freq_customers_churned) / len(high_freq_customers)
    )

    # Compute the churn rate for existing high-spending customers
    churn_rate_existing_high_spending = (
        len(existing_high_spending_customers_churned)
        / len(existing_high_spending_customers)
    )

    # Compute the churn rate for new high-spending customers
    churn_rate_new_high_spending = (
        len(new_high_spending_customers_churned)
        / len(new_high_spending_customers)
    )
    
    # Compute the churn rate for existing high-frequency customers
    churn_rate_existing_high_freq = (
        len(existing_high_freq_customers_churned)
        / len(existing_high_freq_customers)
    )

    # Compute the churn rate for new high-frequency customers
    churn_rate_new_high_freq = (
        len(new_high_freq_customers_churned)
        / len(new_high_freq_customers)
    )

    # Create a DataFrame to store the churn rates among different customer
    # segments
    df_churn_rates = pd.DataFrame(
        [
            [
                churn_rate_existing_high_spending, 
                churn_rate_existing_high_freq, 
                churn_rate_existing,
            ],
            [
                churn_rate_new_high_spending, 
                churn_rate_new_high_freq,
                churn_rate_new,
            ],
            [
                churn_rate_high_spending, 
                churn_rate_high_freq,
                churn_rate_overall,
            ],
        ],
        index=['Existing', 'New', 'Overall'],
        columns=['High_Spending', 'High_Frequency', 'Overall'],
    )
    
    # Create a DataFrame to store the number of churned customers among different 
    # customer segments
    df_cust_churns = pd.DataFrame(
        [
            [
                len(existing_high_spending_customers_churned), 
                len(existing_high_freq_customers_churned),
                len(existing_customers_churned),
            ],
            [
                len(new_high_spending_customers_churned), 
                len(new_high_freq_customers_churned),
                len(new_customers_churned),
            ],
            [
                len(high_spending_customers_churned), 
                len(high_freq_customers_churned),
                len(overall_customers_churned),
            ],
        ],
        index=['Existing', 'New', 'Overall'],
        columns=['High_Spending', 'High_Frequency', 'Overall'],
    )
    
    # Create a DataFrame to store the custom annotaions across different 
    # customer segments for heatmap
    custom_annot = pd.DataFrame(
        [
            [
                f'{churn_rate_existing_high_spending * 100:.1f}%'
                f'\n({len(existing_high_spending_customers_churned)} '
                f'out of {len(existing_high_spending_customers)})',
                f'{churn_rate_existing_high_freq * 100:.1f}%'
                f'\n({len(existing_high_freq_customers_churned)} '
                f'out of {len(existing_high_freq_customers)})',
                f'{churn_rate_existing * 100:.1f}%'
                f'\n({len(existing_customers_churned)} '
                f'out of {len(existing_customers)})',
            ],
            [   
                f'{churn_rate_new_high_spending * 100:.1f}%'
                f'\n({len(new_high_spending_customers_churned)} '
                f'out of {len(new_high_spending_customers)})',
                f'{churn_rate_new_high_freq * 100:.1f}%'
                f'\n({len(new_high_freq_customers_churned)} '
                f'out of {len(new_high_freq_customers)})',
                f'{churn_rate_new * 100:.1f}%'
                f'\n({len(new_customers_churned)} '
                f'out of {len(new_customers)})',
            ],
            [                
                f'{churn_rate_high_spending * 100:.1f}%'
                f'\n({len(high_spending_customers_churned)} '
                f'out of {len(high_spending_customers)})',
                f'{churn_rate_high_freq * 100:.1f}%'
                f'\n({len(high_freq_customers_churned)} '
                f'out of {len(high_freq_customers)})',
                f'{churn_rate_overall * 100:.1f}%'
                f'\n({len(overall_customers_churned)} '
                f'out of {len(overall_customers)})',
            ],
        ],
        index=['Existing', 'New', 'Overall'],
        columns=['High_Spending', 'High_Frequency', 'Overall'],
    )
    
    # Create a DataFrame that summarises the customer churns between 
    # Jul 2022 and Mar 2024
    df_churn_summary = pd.DataFrame(
        [
            [
                'Number of Existing Customers (as of Jul 1, 2022)',
                len(existing_customers),
            ],
            [
                'Number of Churned Existing Customers',
                -len(existing_customers_churned),
            ],
            [
                'Number of New Customers Acquired during the period',
                len(new_customers),
            ],
            [
                'Number of Churned New Customers',
                -len(new_customers_churned),
            ],
            [
                'Number of Retained Customers (as of Mar 2024)',
                len(overall_customers) - len(overall_customers_churned),
            ],
        ],
        columns=['Description', 'count'],
    )
    
    return df_churn_rates, df_cust_churns, df_churn_summary, custom_annot

In [ ]:
def display_churns_heatmap(
        df_churn_rates, custom_annot, df_churn_summary
    ):
    """A function to display customer churns in a heatmap.
    """
    
    plt.figure(figsize=(8, 6))
    
    # Create a heatmap for displaying the customer churns across different 
    # customer groups
    sns.heatmap(
        df_churn_rates, 
        annot=custom_annot, 
        cmap=sns.color_palette('light:#9b442a', as_cmap=True), 
        fmt='',  # This ensures percentage format is respected
        linewidths=0.2,  
        linecolor='gray',
        xticklabels=['High-Spending', 'High-Frequency', 'Overall'],
    )
    
    plt.title('Customer Churns by Segments\n', fontsize=14)
    plt.show()
    
    return df_churn_summary

In [ ]:
# Calculate the churn rates and number of churned customers for different
# customer segments, and summarise customer churns between Jul 2022 and
# Mar 2024
df_churn_rates, df_cust_churns, df_churn_summary, custom_annot = (
    compute_churns(df_cust_segmented)
)

In [ ]:
# Display the churn rates and number of churned customers across different 
# customer segments in heatmap
display_churns_heatmap(
    df_churn_rates, custom_annot, df_churn_summary
)

The **overall churn rate of approximately 52.2% (468 out of 897)** indicates that out of 897 active customers (both existing and new) during the period, 468 (~52.2%) stopped engaging with the café by March 2024, leaving 429 retained customers. 
* **Churn rate among new customers at around 63.9% (360 out of 563)**: During the same period, 360 (~63.9%) out of 563 newly acquired customers stopped engaging with the café.
* **Churn rate among existing customers at around 32.3% (108 out of 334)**: Between July 2022 and March 2024, 108 (~32.3%) out of 334 existing customers (as of Jul 1, 2022) churned.
* **Churn rate among high-spending customers at around 33.9% (56 out of 165)**: Of the 165 high-spending active customers, 56 (~33.9%) churned by the end of the period.
* **Churn rate among high-frequency customers at around 32.9% (53 out of 161)**: During the period, 53 (~32.9%) out of 161 high-frequency customers churned.

These churn rates provide a baseline for evaluating the impact of removing high churn risk items from the menu.

#### Estimate the Reduction in Churn Rate after removing the High Churn Risk Items from the café menu

In [ ]:
# Create a new dataframe that displays each customer, their orders,
# and ordered items after 1st of Jul 2022
df_item_cust = df_items[
    df_items['order_time'] >= start_date
][
    ['order_id', 'item', 'customer_id']
].drop_duplicates().sort_values(
    ['order_id', 'item', 'customer_id']
)

# Create a new dataframe that displays the customers and their orders
# that purchased any of the high churn risk items during the period
df_churn_item_cust = df_item_cust[
    df_item_cust['item'].isin(
        items_high_churn_risk['item']
    )
]

In [ ]:
# Recalculate the churn status assuming they had never placed
# the orders that include the high churn risk items
df_orders = df[
    ['customer_id', 'order_id', 'order_time']
].drop_duplicates().sort_values(
    ['customer_id', 'order_id', 'order_time'],
    ascending=[True, True, True],
).reset_index(drop=True).copy()

high_churn_item_orders = df_churn_item_cust['order_id'].unique()

df_filtered_orders = df_orders[
    ~df_orders['order_id'].isin(
        high_churn_item_orders
    )
]

df_filtered_orders.head()

In [ ]:
# Function to calculate the purchasing interval in days
def get_interval(x):
    if pd.isna(x['next_order_time']):
        return None
    else:
        return (
            x['next_order_time'].date()
            - x['order_time'].date()
        ).days

# Create a function to calculate the revised churn threshold for each customer
def compute_revised_churn_thresholds(df_filtered_orders):
    # Create a new column 'next_order_time' for calculating purchase intervals
    df_filtered_orders['next_order_time'] = df_filtered_orders.sort_values(
        by=['customer_id', 'order_time'], 
        ascending=[True, True],
    ).groupby(['customer_id'])['order_time'].shift(-1)

    # Create new column for recording purchasing interval in days
    df_filtered_orders['purchase_interval_days'] = (
        df_filtered_orders.apply(
            lambda x: get_interval(x), axis=1
        )
    )

    # Calculate the mean of purchasing interval days
    # for each customer
    df_filtered_orders['interval_mean'] = (
        df_filtered_orders.groupby('customer_id')
        ['purchase_interval_days'].transform(np.mean)
    )

    # Calculate the standard deviation of purchasing 
    # interval days for each customer
    df_filtered_orders['interval_std'] = (
        df_filtered_orders.groupby('customer_id')
        ['purchase_interval_days'].transform(np.std)
    )

    # Calculate the revised churn threshold for each customer as 
    # 2 standard deviations above the interval mean
    df_filtered_orders['revised_churn_threshold'] = (
        df_filtered_orders['interval_mean'] + 1 * df_filtered_orders['interval_std']
    )

    # Create a new dataframe that maps the revised churn thresholds 
    # to each customer
    df_revised_churn_thresholds = df_filtered_orders[
        ['customer_id', 'revised_churn_threshold']
    ].drop_duplicates().reset_index(drop=True)

    # Compute 2 standard deviations above the average purchase 
    # interval calculated across the entire customer base
    revised_churn_threshold_all = (
        np.mean(df_filtered_orders['purchase_interval_days'])
        + 1 * np.std(df_filtered_orders['purchase_interval_days'])
    )

    # Impute NaN in 'revised_churn_threshold'
    df_revised_churn_thresholds['revised_churn_threshold'] = (
        df_revised_churn_thresholds['revised_churn_threshold'].apply(
            lambda x: revised_churn_threshold_all if pd.isna(x) else x
        )
    )

    # Append the revised churn threshold to the master dataframe
    df_revised = df.merge(
        df_revised_churn_thresholds,
        how='left',
        on='customer_id',
    )
    
    return df_revised

In [ ]:
# Calculate revised churn threshold for each customer assuming the
# orders that include the high churn risk items had never been placed
df_revised = compute_revised_churn_thresholds(df_filtered_orders)

In [ ]:
df_revised[
    df_revised['churn_threshold'] != df_revised['revised_churn_threshold']
][
    ['customer_id', 'churn_threshold', 'revised_churn_threshold']
].drop_duplicates().head(10)

In [ ]:
churn_analyser = Customer_Churn_Analysis_MX.CustomerChurnAnalyser(df_revised, 'revised_churn_threshold')
df_monthly_churn_revised = churn_analyser.compute_monthly_churn_status()

In [ ]:
cust_start_end_churn_revised = compute_start_end_churn(df_monthly_churn_revised)

In [ ]:
total_spending_threshold_revised, order_freq_threshold_revised, df_spending_freq_revised = (
    compute_spending_freq_thresholds(df_revised)
)

In [ ]:
df_cust_segmented_revised = segment_customers(cust_start_end_churn_revised, df_spending_freq_revised)

In [ ]:
df_churn_rates_revised, df_cust_churns_revised, df_churn_summary_revised, custom_annot_revised = (
    compute_churns(df_cust_segmented_revised)
)

In [ ]:
display_churns_heatmap(
    df_churn_rates_revised, custom_annot_revised, df_churn_summary_revised
)

In [ ]:
df_churn_rates.loc['Overall', 'Overall'] - df_churn_rates_revised.loc['Overall', 'Overall']